# ETL Silver - ECMWF por sub-cuenca (cf / pf)

Agrega el pronostico de Bronze a **una fila por (run_date, run_time, step_hours, miembro, sub-cuenca)**,
promediando los puntos de grilla que caen dentro de cada sub-cuenca.

Dos decisiones que definen esta tabla (Decision 048):

- **Se conservan los 50 miembros del ensemble.** Promediarlos aca destruiria la unica senal de
  incertidumbre que aporta `pf`; el colapso a un numero (media, P90, fraccion sobre umbral) es
  una decision de *Gold*, reversible sin volver a tocar Silver.
- **Se promedian los puntos de cada sub-cuenca.** La media areal es la entrada natural de un
  modelo hidrologico agregado; `n_puntos` viaja en la fila para que se pueda detectar un dia
  con cobertura parcial.

La fuente es **Bronze + el mapa materializado `weather.silver.punto_subcuenca`**, no
`ecmwf_forecast_{cf,pf}_basin`. Ambos caminos tagean el punto con el mismo point-in-polygon; la
diferencia es el costo: `ETL_Silver_ECMWF_PF` lo recalcula con `toPandas()` + geopandas sobre
todas las filas (~1.100 millones en pf, horas de trabajo), mientras que el mapa tiene 436 puntos
y el agregado se resuelve con un JOIN de segundos. `*_basin` sigue siendo la tabla de detalle
punto a punto; esta es la de agregado.

In [ ]:
from datetime import date, timedelta

from delta.tables import DeltaTable
from pyspark.sql import functions as F

# `modelo` elige el par de tablas a procesar. Se corre una vez por modelo (dos tasks en el job)
# en vez de un solo notebook que haga los dos: asi un fallo en pf no arrastra a cf, y cada uno
# reporta su propio conteo.
try:
    dbutils.widgets.dropdown('modelo', 'cf', ['cf', 'pf'])
    dbutils.widgets.dropdown('load_mode', 'incremental', ['full', 'incremental'])
    dbutils.widgets.text('incremental_lookback_days', '5')
    dbutils.widgets.text('max_days_per_run', '400')
    dbutils.widgets.text('chunk_days', '120')
    modelo = dbutils.widgets.get('modelo')
    load_mode = dbutils.widgets.get('load_mode')
    incremental_lookback_days = int(dbutils.widgets.get('incremental_lookback_days'))
    max_days_per_run = int(dbutils.widgets.get('max_days_per_run'))
    chunk_days = int(dbutils.widgets.get('chunk_days'))
except Exception:
    modelo = 'cf'
    load_mode = 'incremental'
    incremental_lookback_days = 5
    max_days_per_run = 400
    chunk_days = 120

if modelo not in ('cf', 'pf'):
    raise ValueError(f"modelo debe ser 'cf' o 'pf', se recibio {modelo!r}")

BRONZE_TABLE = f'weather.bronze.ecmwf_forecast_{modelo}'
BASIN_TABLE = f'weather.silver.ecmwf_forecast_{modelo}_basin'
MAPA_TABLE = 'weather.silver.punto_subcuenca'
TARGET_TABLE = f'weather.silver.ecmwf_forecast_{modelo}_subcuenca'

print(f'modelo={modelo} load_mode={load_mode} lookback={incremental_lookback_days} '
      f'max_days_per_run={max_days_per_run} chunk_days={chunk_days}')
print(f'{BRONZE_TABLE} -> {TARGET_TABLE}')

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TARGET_TABLE} (
  run_date DATE,
  run_time STRING,
  step_hours INT,
  valid_date DATE,
  valid_datetime TIMESTAMP,
  number INT,
  subcuenca_id INT,
  subcuenca_nombre STRING,
  tp_mm_medio DOUBLE,
  n_puntos BIGINT,
  source_table STRING,
  processed_at TIMESTAMP,
  updated_at TIMESTAMP
) USING DELTA
""")

# El mapa punto -> sub-cuenca es el resultado materializado del point-in-polygon que
# ETL_Silver_ECMWF_{CF,PF} resuelve con geopandas. Se deriva de `*_basin` justamente para que
# las dos tablas Silver no puedan divergir: el tageo tiene una sola fuente de verdad.
if not spark.catalog.tableExists(MAPA_TABLE):
    print(f'{MAPA_TABLE} no existe: se construye desde las tablas *_basin')
    partes = [
        spark.table(t).select('latitude', 'longitude', 'subcuenca_id', 'subcuenca_nombre')
        for t in ('weather.silver.ecmwf_forecast_cf_basin', 'weather.silver.ecmwf_forecast_pf_basin')
        if spark.catalog.tableExists(t)
    ]
    if not partes:
        raise ValueError('No hay ninguna tabla *_basin para derivar el mapa punto->sub-cuenca')
    mapa_df = partes[0]
    for extra in partes[1:]:
        mapa_df = mapa_df.unionByName(extra)
    mapa_df = mapa_df.distinct()

    # Un punto que cayera en dos sub-cuencas romperia el promedio areal (se contaria dos veces
    # y `n_puntos` mentiria). El buffer de 0.15 grados de *_basin lo hace posible en teoria, asi
    # que se verifica antes de materializar en vez de descubrirlo como un numero raro en Gold.
    ambiguos = (
        mapa_df.groupBy('latitude', 'longitude').agg(F.countDistinct('subcuenca_id').alias('n'))
        .filter(F.col('n') > 1)
    )
    n_ambiguos = ambiguos.count()
    if n_ambiguos:
        ambiguos.show(20, truncate=False)
        raise ValueError(f'{n_ambiguos} puntos de grilla caen en mas de una sub-cuenca; revisar el buffer de *_basin')

    mapa_df.write.format('delta').mode('overwrite').saveAsTable(MAPA_TABLE)
    print(f'{MAPA_TABLE} creado con {mapa_df.count()} puntos')

mapa = spark.table(MAPA_TABLE).select('latitude', 'longitude', 'subcuenca_id', 'subcuenca_nombre')
puntos_esperados = {r['subcuenca_nombre']: r['n'] for r in mapa.groupBy('subcuenca_nombre').agg(F.count('*').alias('n')).collect()}
print(f'Mapa punto->sub-cuenca: {puntos_esperados}')

In [ ]:
def aggregate_and_merge(dias):
    """Agrega los `dias` indicados (lista de date) y los mergea en Silver.

    El MERGE usa whenMatchedUpdateAll, no solo insert: un dia que se agrego cuando Bronze
    todavia lo tenia incompleto (el sync sube el JSON del dia y Bronze lo carga en corridas
    distintas) tiene que corregirse solo en la proxima pasada, no quedar congelado mal.
    `number` viaja con <=> porque en cf es NULL y `=` nunca lo machearia.
    """
    src = spark.table(BRONZE_TABLE).filter(F.col('run_date').isin(dias))
    agg = (
        src.alias('b')
        .join(
            mapa.alias('p'),
            (F.col('b.latitude') == F.col('p.latitude')) & (F.col('b.longitude') == F.col('p.longitude')),
            'inner',
        )
        .groupBy(
            F.col('b.run_date').alias('run_date'),
            F.col('b.run_time').alias('run_time'),
            F.col('b.step_hours').alias('step_hours'),
            F.col('b.valid_date').alias('valid_date'),
            F.col('b.valid_datetime').alias('valid_datetime'),
            F.col('b.number').alias('number'),
            F.col('p.subcuenca_id').alias('subcuenca_id'),
            F.col('p.subcuenca_nombre').alias('subcuenca_nombre'),
        )
        .agg(
            F.avg(F.col('b.tp_mm')).alias('tp_mm_medio'),
            F.count(F.lit(1)).alias('n_puntos'),
        )
        .withColumn('source_table', F.lit(BRONZE_TABLE))
        .withColumn('processed_at', F.current_timestamp())
        .withColumn('updated_at', F.current_timestamp())
    )

    (
        DeltaTable.forName(spark, TARGET_TABLE).alias('t')
        .merge(
            agg.alias('s'),
            't.run_date = s.run_date AND t.run_time = s.run_time AND t.step_hours = s.step_hours '
            'AND t.number <=> s.number AND t.subcuenca_id = s.subcuenca_id',
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    return len(dias)


def dias_a_procesar():
    """Devuelve la lista de run_date pendientes.

    En incremental no alcanza con "desde el maximo de Silver hacia adelante": el backfill local
    de TIGGE va hacia atras en el tiempo, asi que los dias que llegan nuevos son mas VIEJOS que
    el maximo ya cargado y esa regla no los veria nunca. Por eso el criterio es la diferencia de
    conjuntos (lo que esta en Bronze y no en Silver) mas una ventana corta hacia atras desde el
    maximo, que es la que corrige los dias cargados a medias.
    """
    bronze_dias = {r['run_date'] for r in spark.table(BRONZE_TABLE).select('run_date').distinct().collect()}
    if not bronze_dias:
        return []
    if load_mode == 'full':
        return sorted(bronze_dias)

    silver_dias = {r['run_date'] for r in spark.table(TARGET_TABLE).select('run_date').distinct().collect()}
    faltantes = bronze_dias - silver_dias
    reproceso = set()
    if silver_dias:
        corte = max(silver_dias) - timedelta(days=incremental_lookback_days)
        reproceso = {d for d in bronze_dias if d >= corte}
    return sorted(faltantes | reproceso)


pendientes = dias_a_procesar()
print(f'{len(pendientes)} dias a procesar')
if not pendientes:
    dbutils.notebook.exit(f'{TARGET_TABLE}: nada pendiente')

if len(pendientes) > max_days_per_run:
    # Se cortan los mas viejos primero para que el frente diario nunca quede atras esperando a
    # que termine de drenar el historico.
    print(f'Se limita a los {max_days_per_run} dias mas recientes; el resto queda para la proxima corrida')
    pendientes = pendientes[-max_days_per_run:]

In [ ]:
procesados = 0
for i in range(0, len(pendientes), chunk_days):
    chunk = pendientes[i:i + chunk_days]
    procesados += aggregate_and_merge(chunk)
    print(f'  chunk {chunk[0]} .. {chunk[-1]} ({len(chunk)} dias) OK  [{procesados}/{len(pendientes)}]')

print(f'{procesados} dias agregados en {TARGET_TABLE}')

# Control de cobertura: `n_puntos` tiene que ser exactamente la cantidad de puntos de grilla que
# el mapa asigna a cada sub-cuenca. Un valor menor significa que Bronze tenia el dia incompleto,
# y entonces la media areal de esa fila esta sesgada hacia la parte de la cuenca que si llego.
# Se reporta, no se aborta: hay dias historicos con cobertura parcial conocida y romper el job
# diario por un dia viejo dejaria de publicar los dias sanos.
cobertura = (
    spark.table(TARGET_TABLE)
    .filter(F.col('run_date').isin(pendientes))
    .groupBy('subcuenca_nombre', 'n_puntos')
    .agg(F.countDistinct('run_date').alias('dias'))
    .orderBy('subcuenca_nombre', 'n_puntos')
)
filas = cobertura.collect()
incompletos = [r for r in filas if r['n_puntos'] != puntos_esperados.get(r['subcuenca_nombre'])]
cobertura.show(50, truncate=False)
if incompletos:
    print('AVISO cobertura parcial (n_puntos != esperado):')
    for r in incompletos:
        print(f"  {r['subcuenca_nombre']}: n_puntos={r['n_puntos']} "
              f"(esperado {puntos_esperados.get(r['subcuenca_nombre'])}) en {r['dias']} dia(s)")
else:
    print('Cobertura completa: n_puntos coincide con el mapa en todas las sub-cuencas')

spark.table(TARGET_TABLE).agg(
    F.min('run_date').alias('inicio'),
    F.max('run_date').alias('fin'),
    F.countDistinct('run_date').alias('dias'),
    F.count(F.lit(1)).alias('filas'),
).show(truncate=False)